In [ ]:
import sys
print(sys.executable)   # should end in pharma-rag/.venv/bin/python

### Set up

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "rag"))
from search import search

pd.set_option("display.max_colwidth", 80)
print("Test:", search(keyword="pembrolizumab", k=1)["total_matches"].iloc[0], "pembrolizumab trials")

Test: 345 pembrolizumab trials


In [2]:
TESTS = [
    dict(id=1, type="Concept", question="EGFR inhibitors for lung cancer",
         criterion="Tests an EGFR inhibitor in lung cancer",
         meaning=dict(question="EGFR inhibitor for lung cancer"),
         keyword=dict(keyword="EGFR"),
         hybrid=dict(question="EGFR inhibitor for lung cancer", keyword="EGFR")),
    dict(id=2, type="Concept", question="Immunotherapy for melanoma",
         criterion="Tests an immunotherapy (e.g. checkpoint inhibitor) in melanoma",
         meaning=dict(question="immunotherapy for melanoma"),
         keyword=dict(keyword="melanoma"),
         hybrid=dict(question="immunotherapy for melanoma", keyword="melanoma")),
    dict(id=3, type="Concept", question="Drugs that lower LDL cholesterol",
         criterion="Tests a drug intended to lower LDL cholesterol",
         meaning=dict(question="drugs that lower LDL cholesterol"),
         keyword=dict(keyword="LDL"),
         hybrid=dict(question="drugs that lower LDL cholesterol", keyword="LDL")),
    dict(id=4, type="Concept", question="CAR-T cell therapy for lymphoma",
         criterion="Tests a CAR-T cell therapy in lymphoma",
         meaning=dict(question="CAR-T cell therapy for lymphoma"),
         keyword=dict(keyword="CAR-T"),
         hybrid=dict(question="CAR-T cell therapy for lymphoma", keyword="CAR")),
    dict(id=5, type="Exact name", question="Pembrolizumab trials",
         criterion="Pembrolizumab is one of the interventions",
         meaning=dict(question="pembrolizumab"),
         keyword=dict(keyword="pembrolizumab"),
         hybrid=dict(question="pembrolizumab", keyword="pembrolizumab")),
    dict(id=6, type="Exact name", question="Semaglutide trials",
         criterion="Semaglutide is one of the interventions",
         meaning=dict(question="semaglutide"),
         keyword=dict(keyword="semaglutide"),
         hybrid=dict(question="semaglutide", keyword="semaglutide")),
    dict(id=7, type="Brand name", question="Keytruda trials",
         criterion="Pembrolizumab (Keytruda) is one of the interventions",
         meaning=dict(question="Keytruda"),
         keyword=dict(keyword="Keytruda"),
         hybrid=dict(question="Keytruda", keyword="Keytruda")),
    dict(id=8, type="Brand name", question="Ozempic trials",
         criterion="Semaglutide (Ozempic) is one of the interventions",
         meaning=dict(question="Ozempic"),
         keyword=dict(keyword="Ozempic"),
         hybrid=dict(question="Ozempic", keyword="Ozempic")),
    dict(id=9, type="With filters", question="Recruiting drug trials for Alzheimer's disease",
         criterion="Recruiting, tests a drug, in Alzheimer's disease",
         meaning=dict(question="Alzheimer's disease drug trials recruiting"),
         keyword=dict(keyword="Alzheimer"),
         hybrid=dict(question="Alzheimer's disease", status="RECRUITING",
                     interventional_only=True, drug_only=True)),
    dict(id=10, type="With filters", question="Phase 3 trials for type 2 diabetes",
         criterion="Phase 3, in type 2 diabetes",
         meaning=dict(question="phase 3 trials for type 2 diabetes"),
         keyword=dict(keyword="type 2 diabetes"),
         hybrid=dict(question="type 2 diabetes", phase="Phase 3", interventional_only=True)),
]
METHODS = ["meaning", "keyword", "hybrid"]
print(f"{len(TESTS)} test questions x {len(METHODS)} methods")

10 test questions x 3 methods


### Run searches and create labelling sheet

In [3]:
K = 5
TEST_ID = 1   # change this number to look at another question

t = next(t for t in TESTS if t["id"] == TEST_ID)
print(f"Question: {t['question']}")
print(f"Relevant if: {t['criterion']}\n")

for m in METHODS:
    res = search(**t[m], k=K)
    print(f"=== {m.upper()} ({res['total_matches'].iloc[0] if len(res) else 0} matches) ===")
    for rank, r in enumerate(res.itertuples(), start=1):
        print(f"{rank}. {r.title}")
        print(f"   Conditions: {r.conditions}")
        print(f"   Interventions: {r.interventions}")
        print(f"   {r.phase} | {r.status}\n")

Question: EGFR inhibitors for lung cancer
Relevant if: Tests an EGFR inhibitor in lung cancer



W1116 12:30:11.093000 92522 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== MEANING (1378 matches) ===
1. Study of Safety and Efficacy of EGFR-TKI EGF816 in Combination With cMET Inhibitor INC280 in Adult Patients With EGFR Mutated Non Small Cell Lung Cancer.
   Conditions: Non Small Cell Lung Cancer
   Interventions: Capmatinib, Nazartinib
   Phase 1/2 | TERMINATED

2. A Study Of Combined C- MET Inhibitor And PAN-HER Inhibitor (PF-02341066 And PF-00299804) In Patients With Non- Small Cell Lung Cancer
   Conditions: Non Small Cell Lung Cancer
   Interventions: PF-02341066, PF-00299804
   Phase 1 | COMPLETED

3. Multicenter Study of Rociletinib Administered to Patients With Previously Treated Mutant EGFR Non-small Cell Lung Cancer
   Conditions: Non-small Cell Lung Cancer
   Interventions: Rociletinib
   Phase 2 | TERMINATED

4. A Study of Zipalertinib in Patients With Advanced Non-Small Cell Lung Cancer With Epidermal Growth Factor Receptor (EGFR) Exon 20 Insertions or Other Uncommon Mutation.
   Conditions: Advanced or Metastatic NSCLC Harboring Epidermal

In [4]:
t = next(t for t in TESTS if t["id"] == 1)
meaning_all = search(**t["meaning"], k=2000)
hybrid_all = search(**t["hybrid"], k=2000)
meaning_all["rank"] = range(1, len(meaning_all) + 1)

top = meaning_all.head(len(hybrid_all))
missing = top[~top["trial_id"].isin(hybrid_all["trial_id"])]
print(f"In the meaning top {len(top)}, but not in hybrid: {len(missing)}\n")

for r in missing.itertuples():
    print(f"{r.rank}. ({r.similarity:.2f}) {r.title}")
    print(f"   Interventions: {r.interventions}\n")

In the meaning top 115, but not in hybrid: 58

10. (0.72) AZD9291 Versus Gefitinib or Erlotinib in Patients With Locally Advanced or Metastatic Non-small Cell Lung Cancer
   Interventions: AZD9291 80 mg/40 mg + placebo, Placebo Erlotinib 150/100mg, Placebo Gefitinib 250 mg, Erlotinib 150/100 mg, Gefitinib 250 mg, Placebo AZD9291 80 mg/ 40 mg

20. (0.70) Study of Efficacy and Safety of Nivolumab in Combination With EGF816 and of Nivolumab in Combination With INC280 in Patients With Previously Treated Non-small Cell Lung Cancer
   Interventions: EGF816, INC280, Nivolumab

29. (0.67) Oral Epidermal Growth Factor Receptor Tyrosine Kinase Inhibitors, AZD3759 or AZD9291, in Patients Who Have Advanced Non-Small Cell Lung Cancer
   Interventions: AZD3759, AZD9291

30. (0.66) eXalt3: Study Comparing X-396 (Ensartinib) to Crizotinib in ALK Positive Non-Small Cell Lung Cancer (NSCLC) Patients
   Interventions: X-396 (ensartinib), crizotinib

40. (0.64) ALTA-1L Study: A Study of Brigatinib Versus 

In [5]:
EGFR_WORDS = ["EGFR", "epidermal growth factor receptor",
              "osimertinib", "AZD9291", "erlotinib", "Tarceva", "gefitinib", "Iressa",
              "afatinib", "dacomitinib", "PF-00299804", "nazartinib", "EGF816",
              "rociletinib", "lazertinib", "amivantamab"]

old = search(question="EGFR inhibitor for lung cancer", keyword="EGFR", k=2000)
new = search(question="EGFR inhibitor for lung cancer", keyword=EGFR_WORDS, k=2000)
print(f"Trials found with one word: {len(old)} | with the list: {len(new)}\n")

now_found = missing[missing["trial_id"].isin(new["trial_id"])]
print(f"Of the {len(missing)} trials missed before, now found: {len(now_found)}\n")
for r in now_found.itertuples():
    print(f"{r.rank}. {r.title}")

Trials found with one word: 115 | with the list: 263

Of the 58 trials missed before, now found: 13

10. AZD9291 Versus Gefitinib or Erlotinib in Patients With Locally Advanced or Metastatic Non-small Cell Lung Cancer
20. Study of Efficacy and Safety of Nivolumab in Combination With EGF816 and of Nivolumab in Combination With INC280 in Patients With Previously Treated Non-small Cell Lung Cancer
29. Oral Epidermal Growth Factor Receptor Tyrosine Kinase Inhibitors, AZD3759 or AZD9291, in Patients Who Have Advanced Non-Small Cell Lung Cancer
42. A Study to Learn More About How Well Sevabertinib Works and How Safe it is Compared With Standard Treatment, in Participants Who Have Advanced Non-small Cell Lung Cancer (NSCLC) With Mutations of the Human Epidermal Growth Factor Receptor 2 (HER2)
44. AZD9291 (Osimertinib) Versus Platinum-Based Doublet-Chemotherapy in Locally Advanced or Metastatic Non-Small Cell Lung Cancer
47. A Study Comparing Bevacizumab Therapy With or Without Erlotinib for F

In [7]:
K = 5
rows = []
for t in TESTS:
    for m in METHODS:
        res = search(**t[m], k=K)
        for rank, r in enumerate(res.itertuples(), start=1):
            rows.append(dict(test_id=t["id"], type=t["type"], question=t["question"], method=m,
                             rank=rank, trial_id=r.trial_id, title=r.title,
                             conditions=r.conditions, interventions=r.interventions,
                             phase=r.phase, status=r.status, summary=r.summary))
results = pd.DataFrame(rows, columns=["test_id", "type", "question", "method", "rank", "trial_id",
                                      "title", "conditions", "interventions", "phase", "status", "summary"])
results.to_csv("../eval/retrieval_results.csv", index=False)

criteria = pd.DataFrame(TESTS)[["id", "criterion"]].rename(columns={"id": "test_id"})
to_label = (results.drop_duplicates(["test_id", "trial_id"])
            .merge(criteria, on="test_id")
            [["test_id", "question", "criterion", "trial_id", "title", "conditions",
              "interventions", "phase", "status", "summary"]])
to_label["relevant"] = ""
to_label.to_csv("../eval/retrieval_labels.csv", index=False)

print(f"Results: {len(results)} rows | To label: {len(to_label)} unique question-trial pairs")

Results: 134 rows | To label: 100 unique question-trial pairs


In [8]:
from rewrite import rewrite

question = "EGFR inhibitors for lung cancer"
words = rewrite(question)
print("LLM words:", words, "\n")

found = search(question=question, keyword=words, k=2000)
now_found = missing[missing["trial_id"].isin(found["trial_id"])]
print(f"Trials found: {len(found)}")
print(f"Of the {len(missing)} trials missed before, now found: {len(now_found)}\n")
for r in now_found.itertuples():
    print(f"{r.rank}. {r.title}")

LLM words: ['epidermal growth factor receptor', 'EGFR', 'erlotinib', 'gefitinib', 'afatinib', 'osimertinib', 'dacomitinib', 'icotinib', 'mobocertinib', 'lazertinib', 'AZD9291'] 

Trials found: 261
Of the 58 trials missed before, now found: 12

10. AZD9291 Versus Gefitinib or Erlotinib in Patients With Locally Advanced or Metastatic Non-small Cell Lung Cancer
29. Oral Epidermal Growth Factor Receptor Tyrosine Kinase Inhibitors, AZD3759 or AZD9291, in Patients Who Have Advanced Non-Small Cell Lung Cancer
42. A Study to Learn More About How Well Sevabertinib Works and How Safe it is Compared With Standard Treatment, in Participants Who Have Advanced Non-small Cell Lung Cancer (NSCLC) With Mutations of the Human Epidermal Growth Factor Receptor 2 (HER2)
44. AZD9291 (Osimertinib) Versus Platinum-Based Doublet-Chemotherapy in Locally Advanced or Metastatic Non-Small Cell Lung Cancer
47. A Study Comparing Bevacizumab Therapy With or Without Erlotinib for First-Line Treatment of Non-Small Cell

In [ ]:
question = "Keytruda trials"
words = rewrite(question)
print("LLM words:", words, "\n")

for label, res in [("Without LLM", search(keyword="Keytruda", k=5)),
                   ("With LLM", search(question=question, keyword=words, k=5))]:
    print(f"=== {label} ({res['total_matches'].iloc[0] if len(res) else 0} matches) ===")
    for r in res.itertuples():
        print(f"- {r.title}\n  Interventions: {r.interventions}")
    print()

AttributeError: 'list' object has no attribute 'items'

: 